In [9]:
import pandas as pd
import sys
import os

# Add parent directory to path so we can import src modules
sys.path.append(os.path.abspath('..'))


In [2]:
df = pd.read_csv("../data/processed/engineered_dataset.csv")

df.head()

,failed_connection_attempts,packets_sent_per_sec,packets_recv_per_sec,tcp_connection_count,context_switch_rate,unique_dst_port_count,top_process_cpu_pct,cpu_user_pct,load_avg_1m,attack_type,is_attack,packet_rate_total,packet_spike_ratio,connection_spike_ratio,cpu_spike_ratio,process_cpu_ratio,failure_rate,port_scan_intensity,load_per_connection
0,0.0,0.0,0.0,9.0,567.0,0,0.0,6.21,0.18,crypto,1,0.0,1.002004,1.000000,1.000000,0.000000,0.0,0.0,0.018000
1,0.0,0.0,0.0,9.0,803.0,0,0.0,1.50,0.41,crypto,1,0.0,1.002004,1.000000,0.389105,0.000000,0.0,0.0,0.041000
2,0.0,0.0,0.0,10.0,893.0,0,0.0,5.68,0.41,crypto,1,0.0,1.002004,1.071429,1.272591,0.000000,0.0,0.0,0.037273
3,0.0,0.0,0.0,14.0,404.0,0,26.0,4.99,0.41,crypto,1,0.0,1.002004,1.333333,1.085963,5.210410,0.0,0.0,0.027333
4,0.0,0.0,0.0,14.0,496.0,0,59.9,2.17,0.37,crypto,1,0.0,1.002004,1.250000,0.527981,27.603559,0.0,0.0,0.024667


In [6]:
from src.anomaly_model import AnomalyDetector
from src.classification_model import AttackClassifier

# Load anomaly model
anom = AnomalyDetector()
anom.load("../outputs/models/anomaly_model.pkl")

# Load classifier
clf = AttackClassifier()
clf.load("../outputs/models/classifier.pkl")

In [10]:
anomaly_preds, scores = anom.predict(df)

df['anomaly_pred'] = anomaly_preds

In [11]:
df['attack_pred'] = clf.predict(df)

In [12]:
df['final_prediction'] = df.apply(
    lambda row: row['attack_pred'] if row['anomaly_pred'] == 1 else 'none',
    axis=1
)

In [13]:
from sklearn.metrics import classification_report, confusion_matrix

print("Confusion Matrix:\n")
print(confusion_matrix(df['attack_type'], df['final_prediction']))

print("\nClassification Report:\n")
print(classification_report(df['attack_type'], df['final_prediction']))

Confusion Matrix:

[[ 497    0    3    0]
 [   0  530   70    0]
 [   0    0 3900    0]
 [   0    0    0  600]]

Classification Report:

              precision    recall  f1-score   support

      crypto       1.00      0.99      1.00       500
        ddos       1.00      0.88      0.94       600
        none       0.98      1.00      0.99      3900
    portscan       1.00      1.00      1.00       600

    accuracy                           0.99      5600
   macro avg       1.00      0.97      0.98      5600
weighted avg       0.99      0.99      0.99      5600



In [14]:
print("Anomaly Detection Only:\n")
print(df.groupby('attack_type')['anomaly_pred'].mean())

print("\nHybrid Final Output:\n")
print(df.groupby('attack_type')['final_prediction'].value_counts())

Anomaly Detection Only:

attack_type
crypto      0.996000
ddos        0.883333
none        0.216923
portscan    1.000000
Name: anomaly_pred, dtype: float64

Hybrid Final Output:

attack_type  final_prediction
crypto       crypto               497
             none                   3
ddos         ddos                 530
             none                  70
none         none                3900
portscan     portscan             600
Name: count, dtype: int64
